# 🚗 Driver Behavior Anomaly Detection
## Notebook 00 — Setup & Configuration

This project classifies driving sessions into **3 categories**:
| Class | Meaning |
|-------|---------|
| 🟢 NORMAL | Smooth, safe driving |
| 🔴 AGGRESSIVE | Harsh braking, sharp cornering, rapid acceleration |
| 🔵 SLOW | Drowsy or overly cautious driving |

### Pipeline Overview
```
00_Setup → 01_Exploration → 02_Preprocessing → 03_Windowing
         → 04_Balance_Normalize → 05_Model_Training → 06_Evaluation
```

**Dataset used:** Your uploaded sensor CSV (accelerometer + gyroscope readings)  
For larger experiments, download the [Kaggle Driving Behavior Dataset](https://www.kaggle.com/datasets/outofskills/driving-behavior)

In [ ]:
# Install required libraries (run once)
# !pip install tensorflow scikit-learn imbalanced-learn pandas numpy matplotlib seaborn

In [1]:
# ── Standard imports used across all notebooks ──────────────
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Core libraries ready")
print(f"   NumPy    : {np.__version__}")
print(f"   Pandas   : {pd.__version__}")

✅ Core libraries ready
   NumPy    : 2.1.3
   Pandas   : 2.2.3


In [2]:
# ── Project-wide configuration ───────────────────────────────

# Paths
DATA_LABELED   = "test_motion_data.csv"
DATA_UNLABELED = "test_motion_data_nolabels.csv"
OUTPUT_DIR     = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sensor columns (raw + derived)
SENSOR_COLS  = ["AccX", "AccY", "AccZ", "GyroX", "GyroY", "GyroZ"]
DERIVED_COLS = ["jerk_x", "jerk_y", "jerk_z", "lateral_qforce", "speed_variance"]
ALL_FEATURES = SENSOR_COLS + DERIVED_COLS   # 11 total channels

# Class labels
CLASS_MAP   = {"NORMAL": 0, "AGGRESSIVE": 1, "SLOW": 2}
CLASS_NAMES = ["NORMAL", "AGGRESSIVE", "SLOW"]
CLASS_COLORS = {"NORMAL": "#2ecc71", "AGGRESSIVE": "#e74c3c", "SLOW": "#3498db"}
NUM_CLASSES  = 3

# Windowing
WINDOW_SIZE    = 50    # ~5 seconds at ~10 Hz
WINDOW_STRIDE  = 25    # 50% overlap
SAMPLING_RATE  = 10    # Hz

# Model
CNN_FILTERS  = 64
LSTM_UNITS   = 64
DROPOUT      = 0.3
DENSE_UNITS  = 64

# Training
EPOCHS       = 60
BATCH_SIZE   = 64
VAL_SPLIT    = 0.15
TEST_SPLIT   = 0.15
RANDOM_SEED  = 42

# Class weights (penalise aggressive & slow more)
CLASS_WEIGHTS = {0: 1.0, 1: 2.5, 2: 2.5}

print("✅ Configuration set")
print(f"   Sensor channels : {len(SENSOR_COLS)} raw + {len(DERIVED_COLS)} derived = {len(ALL_FEATURES)} total")
print(f"   Window size     : {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLING_RATE} seconds)")
print(f"   Classes         : {CLASS_NAMES}")

✅ Configuration set
   Sensor channels : 6 raw + 5 derived = 11 total
   Window size     : 50 samples (5.0 seconds)
   Classes         : ['NORMAL', 'AGGRESSIVE', 'SLOW']
